# Unidad 23. Eficiencia, transformaciones y texto

**Duración orientativa:** 7 horas · **UT3**

Esta unidad cierra el bloque de Pandas antes del proyecto integrador. Aprenderás a medir, transformar columnas completas y procesar texto de forma clara y reproducible.

## Resultados de aprendizaje

Al finalizar podrás medir memoria, elegir tipos con prudencia, preferir operaciones vectorizadas, decidir cuándo map o apply son adecuados y usar expresiones regulares verificables.

**Material:** datos/pandas/20_lecturas_estaciones_limpias.csv.

## 1. Eficiencia con criterio

La prioridad es código correcto, legible y comprobable. Solo después se mide si existe un problema de tiempo o memoria. Pandas trabaja principalmente sobre columnas completas: una operación vectorizada expresa una misma regla para todos los valores y suele ser más rápida que un bucle por filas.

No optimices sin medir. Un código corto pero opaco puede ser peor que uno algo más lento y auditable.

In [ ]:
from pathlib import Path
import pandas as pd
ruta=Path('..')/'datos'/'pandas'/'20_lecturas_estaciones_limpias.csv'
if not ruta.exists(): ruta=Path('datos')/'pandas'/'20_lecturas_estaciones_limpias.csv'
lecturas=pd.read_csv(ruta)
lecturas.head()

## 2. Memoria y tipos

memory_usage con deep=True estima memoria, incluido texto. category puede ahorrar memoria cuando pocos textos se repiten muchas veces, como zona. No es recomendable para identificadores casi únicos. Reducir números exige comprobar rango, precisión y significado.

In [ ]:
antes=lecturas.memory_usage(deep=True)
optimizada=lecturas.copy()
optimizada['zona']=optimizada['zona'].astype('category')
print(antes)
print('Zona antes:',antes['zona'])
print('Zona después:',optimizada['zona'].memory_usage(deep=True))
print('Valores únicos:',lecturas['zona'].nunique())

## 3. Transformaciones vectorizadas

Una operación sobre una Series transforma todos sus valores sin recorrer filas de forma explícita. Nombra las nuevas columnas con su unidad o significado. Los límites de clasificación deben quedar documentados; los de este cuaderno son didácticos, no normas sanitarias.

In [ ]:
optimizada['temperatura_kelvin']=optimizada['temperatura_c']+273.15
optimizada['nivel_pm25']=pd.cut(optimizada['pm25_ug_m3'],[-float('inf'),15,20,float('inf')],labels=['baja','media','alta'])
optimizada[['temperatura_c','temperatura_kelvin','pm25_ug_m3','nivel_pm25']].head()

## 4. map, apply y filas

map transforma valores de una Series mediante un diccionario o una función sencilla. apply sobre una Series puede ser útil si no existe un método vectorizado claro. apply(axis=1) entrega una fila completa a una función y suele ser más lento; antes de usarlo busca operaciones aritméticas, where, select, merge o métodos str.

Una función aplicada debe ser pequeña, tener un nombre expresivo y contemplar valores ausentes o límite.

In [ ]:
optimizada['turno']=optimizada['hora'].map(lambda hora:'mañana' if hora<'12:00' else 'tarde')
def etiqueta(valor):
    return 'revisar' if valor>=20 else 'ordinaria'
optimizada['etiqueta']=optimizada['pm25_ug_m3'].apply(etiqueta)
optimizada[['hora','turno','pm25_ug_m3','etiqueta']].head()

## 5. Texto y expresiones regulares

Los métodos str operan sobre una Series de texto: strip elimina espacios, lower normaliza mayúsculas, contains busca y extract recupera partes. Una regex debe definirse mediante ejemplos válidos e inválidos. ^ marca inicio, $ final, \d un dígito y {3} exactamente tres repeticiones.

fullmatch valida toda la cadena; contains acepta una coincidencia parcial. Para validar un identificador completo, fullmatch suele ser la opción correcta.

In [ ]:
patron=r'^L-\d{3}$'
optimizada['id_valido']=optimizada['id_lectura'].str.fullmatch(patron).notna()
optimizada['numero_lectura']=optimizada['id_lectura'].str.extract(r'(\d+)$',expand=False).astype(int)
optimizada[['id_lectura','id_valido','numero_lectura']].head()

## 6. Comprobación y trazabilidad

Después de transformar, comprueba tipos, ausencias y valores esperados. Conserva el DataFrame original para comparar. En un proyecto debe quedar documentada la versión de las bibliotecas, el origen de datos y la regla de cada transformación.

In [ ]:
assert lecturas['zona'].dtype==object
assert str(optimizada['zona'].dtype)=='category'
assert optimizada['temperatura_kelvin'].notna().all()
assert optimizada['id_valido'].all()
print('Comprobaciones superadas')

## Errores frecuentes

- Convertir tipos sin comprobar precisión o cardinalidad.
- Usar apply(axis=1) cuando una operación vectorizada expresa la regla.
- Usar una regex demasiado permisiva.
- Convertir errores a ausencias y no revisarlas.
- Presentar una clasificación didáctica como recomendación real.

## Práctica guiada

1. Compara la memoria de zona antes y después de category.
2. Cambia el umbral de nivel_pm25 y observa el resultado.
3. Prueba identificadores válidos e inválidos contra el patrón.
4. Cambia fullmatch por contains y explica la diferencia.
5. Identifica una operación que no debería usar apply(axis=1).

## Ejercicios

**23.01.** Mide memoria de estacion_id y zona. Justifica cuál convertirías a category.

**23.02.** Crea temperatura_fahrenheit de forma vectorizada y comprueba la primera fila.

**23.03.** Crea humedad_estado: baja si humedad_pct < 55, alta si > 70 y normal en el resto, sin apply(axis=1).

**23.04.** Extrae el número de estacion_id, conviértelo a entero y valida el patrón EST- seguido de dos dígitos.

**23.05.** Crea una Series con comentarios de sensores, normaliza espacios y mayúsculas, y extrae códigos INC- seguidos de tres dígitos. Explica qué ocurre sin coincidencia.

**23.06.** Propón una transformación vectorizada para el proyecto Spotify y explica por qué evitarías apply por fila.

In [ ]:
# 23.01. Mide memoria de estacion_id y zona. Justifica cuál convertirías a category.
print(lecturas['zona'].memory_usage(deep=True))
print(lecturas['estacion_id'].memory_usage(deep=True))
print(lecturas['zona'].nunique())
print(lecturas['estacion_id'].nunique())

# Convertiria las dos a category ya que tiene pocos valores repetidos muchas veces.

In [ ]:
# 23.02. Crea temperatura_fahrenheit de forma vectorizada y comprueba la primera fila.
optimizada['temperatura_fahrenheit'] = optimizada['temperatura_c']*1.8+32
print(optimizada.head(1))

In [ ]:
# 23.03. Crea humedad_estado: baja si humedad_pct < 55, alta si > 70 y normal en el resto, sin apply(axis=1).
import numpy as np
optimizada['humedad_estado'] = np.where(optimizada['humedad_pct'] < 55, 'baja', np.where(optimizada['humedad_pct'] > 70, 'alta', 'normal'))

In [ ]:
patron=r'^L-\d{3}$'
optimizada['id_valido']=optimizada['id_lectura'].str.fullmatch(patron).notna()
optimizada['numero_lectura']=optimizada['id_lectura'].str.extract(r'(\d+)$',expand=False).astype(int)
optimizada[['id_lectura','id_valido','numero_lectura']].head()

In [19]:
# 23.04. Extrae el número de estacion_id, conviértelo a entero y valida el patrón EST- seguido de dos dígitos. EST-01
patron = r'EST-\d{2}'
optimizada['id_estacion_valida'] = optimizada['estacion_id'].str.fullmatch(patron).notna()
optimizada['numero_estacion'] = optimizada['estacion_id'].str.extract(r'(\d+)$', expand=False).astype(int)


In [21]:
# 23.05. Crea una Series con comentarios de sensores, normaliza espacios y mayúsculas, y extrae códigos INC- seguidos de tres dígitos. Explica qué ocurre sin coincidencia.
comentarios = pd.Series([
    '  inc-123 sensor reiniciado  ',
    'INC-456   batería baja',
    'sin incidencia',
    '  Inc-789  calibrado ',
    None
])
comentarios_limpios = (
    comentarios
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.upper()
)
codigos = comentarios_limpios.str.extract(r'(INC-\d{3})')

# str.extract() devuelve NaN para las filas donde no encuentra el patrón.

In [ ]:
# 23.06. Propón una transformación vectorizada para el proyecto Spotify y explica por qué evitarías apply por fila

## Autoevaluación

¿Puedes explicar qué medir antes de optimizar, cuándo category puede ayudar, qué alternativa revisar antes de apply(axis=1), qué valida fullmatch y qué comprobaciones dejar tras una transformación?

Con esta unidad queda preparado el proyecto integrador de Pandas.